In [1]:
import numpy as np
from tensorflow.keras.datasets import reuters
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
(X_train, _), (_,_) = reuters.load_data(num_words=5000)
X_train = X_train[:1000]

word_index = reuters.get_word_index()

reverse_word_index = {v+3 : k for k, v in word_index.items()}


2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
550378/550378 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
decoded_texts = []
for sequence in X_train:
  text = " ".join([reverse_word_index.get(i, "?") for i in sequence])
  decoded_texts.append(text)


In [4]:
input_texts = []
decoder_inputs = []
decoder_targets = []

for text in decoded_texts:
  words = text.split()
  input_texts.append(text)

  summary = " ".join(words[:10])
  decoder_inputs.append('<sos>'  + summary)
  decoder_targets.append(summary + ' <eos>')

tokenizer = Tokenizer(num_words=5000, filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
tokenizer.fit_on_texts(input_texts + decoder_inputs + decoder_targets)
vocab_size = len(tokenizer.word_index)+1

encoder_seq = tokenizer.texts_to_sequences(input_texts)
decoder_inp_seq = tokenizer.texts_to_sequences(decoder_inputs)
decoder_tgt_seq = tokenizer.texts_to_sequences(decoder_targets)

# Pas sequences
max_enc_len = 50
max_dec_len = 12 # 10 words + 1 tag

encoder_input_data = pad_sequences(encoder_seq, maxlen=max_enc_len, padding='pre')
decoder_input_data = pad_sequences(decoder_inp_seq, maxlen=max_dec_len, padding='post')
decoder_target_data = pad_sequences(decoder_tgt_seq, maxlen=max_dec_len, padding='post')

In [20]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding

# creating encoder
encoder_inputs = Input(shape=(max_enc_len,))
enc_emb = Embedding(vocab_size, 100, input_length=max_enc_len)(encoder_inputs)
encoder_lstm = LSTM(128, return_state=True)

encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# creating decoder
decoder_inputs = Input(shape=(max_dec_len,))
dec_emb_layer = Embedding(vocab_size, 100, input_length=max_dec_len)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)

decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(vocab_size, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, epochs=15, batch_size=32)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_14      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_15      │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 50, 100)   │    470,100 │ input_layer_14[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 12, 100)   │    470,100 │ input_layer_15[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ [(None, 128),     │    117,248 │ embedding_8[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ [(None, 12, 128), │    117,248 │ embedding_9[0][0… │
│                     │ (None, 128),      │            │ lstm_8[0][1],     │
│                     │ (None, 128)]      │            │ lstm_8[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 12, 4701)  │    606,429 │ lstm_9[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,781,125 (6.79 MB)

 Trainable params: 1,781,125 (6.79 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 7.2368
Epoch 2/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 4.8393
Epoch 3/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 4.5580
Epoch 4/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 4.4032
Epoch 5/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 4.2708
Epoch 6/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 4.1499
Epoch 7/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 4.0459
Epoch 8/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 3.9526
Epoch 9/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.8575
Epoch 10/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 3.7666
Epoch 11/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.6803
Epoch 12/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.5952
Epoch 13/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.5192
Epoch 14/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 3.4518
Epoch 15/15
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 3.3916


In [21]:
# encoder standalone model
encoder_model = Model(encoder_inputs, encoder_states)

# decoder standalone model
decoder_state_input_h = Input(shape=(128,))
decoder_state_input_c = Input(shape=(128,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb_inf = dec_emb_layer(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(dec_emb_inf, initial_state=decoder_states_inputs)
decoder_states_inf = [state_h_inf, state_c_inf]

decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs, # inputs
    [decoder_outputs_inf] + decoder_states_inf # outputs/targets
)

In [22]:
# text generation function
def generate_summary(input_seq):
  # get the context vector from encoder
  states_value = encoder_model.predict(input_seq)

  # start decoder with the <sos> token
  target_seq = np.zeros((1,1))
  target_seq[0, 0] = tokenizer.word_index['<sos>']

  stop_condition = False
  decoded_sentence = ""

  while not stop_condition:
    output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
    sampled_token_index = np.argmax(output_tokens[0, -1, :])

    if sampled_token_index == 0:
      break
    sampled_word = tokenizer.index_word[sampled_token_index]

    if sampled_word == '<eos>' or len(decoded_sentence.split()) > max_dec_len:
      stop_condition = True
    else:
      decoded_sentence += " " + sampled_word

    target_seq = np.zeros((1,1))
    target_seq[0,0] = sampled_token_index

    states_value = [h, c]

  return decoded_sentence.strip()

In [23]:
print("\n--- EVALUATION ---")
for i in range(3):
  print(f"\nOriginal Text: {input_texts[i][:50]}...")
  print(f"Actual Summary: {decoder_targets[i].replace(' <eos>','')}")
  print(f"Predicted Summary: {generate_summary(encoder_input_data[i:i+1])}")


--- EVALUATION ---

Original Text: ? ? ? said as a result of its december acquisition...
Actual Summary: ? ? ? said as a result of its december
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
Predicted Summary: the corp said it has has to

Original Text: ? generale de banque sa lt ? ? and lt heller overs...
Actual Summary: ? generale de banque sa lt ? ? and lt
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Predicted Summary: the corp said it has has to

Original Text: ? shr 3 28 dlrs vs 22 cts shr diluted 2 99 dlrs vs...
Actual Summary: ? shr 3 28 dlrs vs 22 cts shr diluted
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Predicted Summary: shr loss cts vs loss cts net 1
